# Los Cumbres For The People: quickstart

This notebook drives an LCFTP observatory client: it shows your observatory,
asks the server for an observation plan, runs it with your configured
telescope driver, and uploads the frames.

Before you start, register your observatory on the LCFTP web site, create a
device token, and run `lcftp-client init --server URL --token TOKEN` (or set
`LCFTP_CLIENT_CONFIG` to the path of an existing client configuration).

Everything below uses top-level `await`, which works in JupyterLab and on the
Rubin Science Platform.

## Settings

In [ ]:
import os
from datetime import UTC, datetime, timedelta

from lcftp_client.clock import FakeClock, SystemClock
from lcftp_client.config import load_config
from lcftp_client.session import LcftpClient

# When the observing window starts (default: now) and how long it lasts.
START = datetime.fromisoformat(
    os.environ.get("LCFTP_NOTEBOOK_START") or datetime.now(tz=UTC).isoformat()
)
HOURS = 10

# Fast-forward through the plan instead of waiting for each target's time.
# Only sensible with the simulated driver.
FAST_FORWARD = os.environ.get("LCFTP_NOTEBOOK_FAST_FORWARD") == "1"

config = load_config()
clock = FakeClock(START) if FAST_FORWARD else SystemClock()
client = LcftpClient(config, clock=clock)
print(f"Observatory {config.observatory_id}, driver {config.driver}")

## Your observatory

In [ ]:
profile = await client.profile()
site = profile.site
print(
    f"{profile.name}: {site.latitude:.2f}, {site.longitude:.2f}, "
    f"{site.elevation_m:.0f} m ({site.timezone})"
)
print(
    f"Aperture {profile.aperture_m} m, limiting magnitude {profile.limiting_mag}"
)

## Request a plan

In [ ]:
from astropy.table import Table

plan = await client.request_plan(START, START + timedelta(hours=HOURS))
print(
    f"Plan {plan.plan_id}: {len(plan.items)} targets "
    f"(candidates as of {plan.candidates_as_of})"
)
Table(
    rows=[
        (
            item.item_id,
            item.candidate.dia_object_id,
            round(item.candidate.ra, 4),
            round(item.candidate.dec, 4),
            round(item.candidate.mag, 2),
            item.not_before.strftime("%H:%M"),
            item.not_after.strftime("%H:%M"),
        )
        for item in plan.items
    ],
    names=(
        "item",
        "diaObjectId",
        "ra",
        "dec",
        "mag",
        "start (UTC)",
        "end (UTC)",
    ),
)

## Altitude of each target

In [ ]:
import astropy.units as u
import numpy as np
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time
from astropy.utils import iers

iers.conf.auto_download = False

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed; skipping the plot")

if plt is not None and plan.items:
    location = EarthLocation.from_geodetic(
        site.longitude * u.deg, site.latitude * u.deg, site.elevation_m * u.m
    )
    times = Time(START) + np.linspace(0, HOURS, 100) * u.hour
    frame = AltAz(obstime=times, location=location)
    fig, ax = plt.subplots(figsize=(8, 4))
    for item in plan.items:
        coord = SkyCoord(item.candidate.ra * u.deg, item.candidate.dec * u.deg)
        ax.plot(
            times.datetime,
            coord.transform_to(frame).alt.deg,
            label=str(item.candidate.dia_object_id),
        )
        ax.axvspan(item.not_before, item.not_after, alpha=0.1)
    ax.axhline(
        profile.constraints.min_altitude_deg, color="gray", linestyle="--"
    )
    ax.set_ylim(0, 90)
    ax.set_xlabel("UTC")
    ax.set_ylabel("Altitude (deg)")
    ax.legend(fontsize="small", ncol=2)
    plt.show()

## Observe

This runs the plan with your telescope driver. Interrupt the kernel or call `client.stop()` from another cell to stop; the telescope is parked either way.

In [ ]:
outcomes = await client.run(plan)
for outcome in outcomes:
    print(
        outcome.item_id,
        outcome.status,
        outcome.reason.value,
        outcome.detail or "",
    )

## Upload

In [ ]:
summary = await client.upload()
summary

In [ ]:
client.status()

In [ ]:
await client.server.aclose()